# 1. Producing the data  
In this task, we will implement Apache Kafka producers to simulate real-time data streaming. Spark and parallel data processing should not be used in this section, as we are simulating sensors that often lack processing capabilities.  

1. Every 1 second, load 50-100 accident records from the CSV file. You should keep a pointer in the file reading process and advance it per read. The data should be read in chronological order; just the number of records you read every second is a random number between 50 and 100 (inclusive).
2. Add the current timestamp (accident_ts) to the records. 
3. Send your batch of accident data to a Kafka topic with an appropriate name.

In [1]:
# This producer simulates real-time accident sensor data.
# Every 1 second, it reads 50-100 records in chronological order,
# adds the current timestamp as accident_ts, and sends the batch to Kafka.
#
# Important:
# Spark and parallel processing are not used in this task.
# accident_ts is sent as a numeric Unix timestamp, as required by Task 2.
# The CSV file is read using one csv.DictReader object, so the file pointer
# naturally moves forward and records are sent in chronological order.

import csv
import json
import random
import datetime as dt
from time import sleep
from kafka import KafkaProducer


# Configuration
hostip = "kafka"              
topic = "accident_stream"     # Kafka topic for accident streaming data
csv_file = "A2B/streaming_collision.csv"


# Helper function: connect to Kafka producer
def connect_kafka_producer():
    """
    Create and return a KafkaProducer instance.
    The value_serializer converts Python objects into JSON bytes.
    """
    producer = None

    try:
        producer = KafkaProducer(
            bootstrap_servers=[f"{hostip}:9092"],
            value_serializer=lambda x: json.dumps(x).encode("utf-8"),
            api_version=(3, 9)
        )
        print("Kafka producer connected successfully.")

    except Exception as ex:
        print("Exception while connecting Kafka producer:")
        print(str(ex))

    finally:
        return producer


# Helper function: publish one batch message to Kafka
def publish_message(producer_instance, topic_name, key, value):
    """
    Send one batch of accident records to the Kafka topic.
    """
    try:
        key_bytes = bytes(key, encoding="utf-8")

        producer_instance.send(
            topic_name,
            key=key_bytes,
            value=value
        )

        producer_instance.flush()

        print(
            f"Published batch '{key}' with {len(value)} records "
            f"to topic '{topic_name}'."
        )

    except Exception as ex:
        print("Exception in publishing message:")
        print(str(ex))


# Main producer logic
producer = connect_kafka_producer()

if producer is not None:
    with open(csv_file, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        batch_id = 0
        end_of_file = False

        while not end_of_file:
            # Randomly choose how many records to read this second
            batch_size = random.randint(50, 100)

            records = []

            # Read the next 50-100 rows from the file.
            # Because we keep using the same reader object,
            # this acts as a file pointer and preserves chronological order.
            for _ in range(batch_size):
                try:
                    row = next(reader)

                except StopIteration:
                    print("End of CSV file reached. Stopping producer.")
                    end_of_file = True
                    break

                # Add current timestamp as a numeric Unix timestamp.
                row["accident_ts"] = int(dt.datetime.now().timestamp())

                records.append(row)

            # Only publish if the current batch contains records.
            # This handles the final partial batch before the file ends.
            if len(records) > 0:
                batch_id += 1

                publish_message(
                    producer_instance=producer,
                    topic_name=topic,
                    key=f"batch_{batch_id}",
                    value=records
                )

                # Wait 1 second before sending the next batch
                sleep(1)

        # Close producer cleanly after all records have been sent
        producer.close()
        print("Kafka producer closed.")

Kafka producer connected successfully.
Published batch 'batch_1' with 72 records to topic 'accident_stream'.
Published batch 'batch_2' with 97 records to topic 'accident_stream'.
Published batch 'batch_3' with 60 records to topic 'accident_stream'.
Published batch 'batch_4' with 76 records to topic 'accident_stream'.
Published batch 'batch_5' with 54 records to topic 'accident_stream'.
Published batch 'batch_6' with 94 records to topic 'accident_stream'.
Published batch 'batch_7' with 97 records to topic 'accident_stream'.
Published batch 'batch_8' with 82 records to topic 'accident_stream'.
Published batch 'batch_9' with 68 records to topic 'accident_stream'.
Published batch 'batch_10' with 50 records to topic 'accident_stream'.
Published batch 'batch_11' with 94 records to topic 'accident_stream'.
Published batch 'batch_12' with 64 records to topic 'accident_stream'.
Published batch 'batch_13' with 78 records to topic 'accident_stream'.
Published batch 'batch_14' with 61 records to t

Published batch 'batch_116' with 59 records to topic 'accident_stream'.
Published batch 'batch_117' with 65 records to topic 'accident_stream'.
Published batch 'batch_118' with 56 records to topic 'accident_stream'.
Published batch 'batch_119' with 52 records to topic 'accident_stream'.
Published batch 'batch_120' with 90 records to topic 'accident_stream'.
Published batch 'batch_121' with 80 records to topic 'accident_stream'.
Published batch 'batch_122' with 64 records to topic 'accident_stream'.
Published batch 'batch_123' with 65 records to topic 'accident_stream'.
Published batch 'batch_124' with 50 records to topic 'accident_stream'.
Published batch 'batch_125' with 98 records to topic 'accident_stream'.
Published batch 'batch_126' with 65 records to topic 'accident_stream'.
Published batch 'batch_127' with 62 records to topic 'accident_stream'.
Published batch 'batch_128' with 89 records to topic 'accident_stream'.
Published batch 'batch_129' with 99 records to topic 'accident_s

Published batch 'batch_230' with 69 records to topic 'accident_stream'.
Published batch 'batch_231' with 86 records to topic 'accident_stream'.
Published batch 'batch_232' with 50 records to topic 'accident_stream'.
Published batch 'batch_233' with 85 records to topic 'accident_stream'.
Published batch 'batch_234' with 84 records to topic 'accident_stream'.
Published batch 'batch_235' with 81 records to topic 'accident_stream'.
Published batch 'batch_236' with 67 records to topic 'accident_stream'.
Published batch 'batch_237' with 86 records to topic 'accident_stream'.
Published batch 'batch_238' with 100 records to topic 'accident_stream'.
Published batch 'batch_239' with 71 records to topic 'accident_stream'.
Published batch 'batch_240' with 79 records to topic 'accident_stream'.
Published batch 'batch_241' with 74 records to topic 'accident_stream'.
Published batch 'batch_242' with 72 records to topic 'accident_stream'.
Published batch 'batch_243' with 72 records to topic 'accident_

Published batch 'batch_344' with 100 records to topic 'accident_stream'.
Published batch 'batch_345' with 51 records to topic 'accident_stream'.
Published batch 'batch_346' with 97 records to topic 'accident_stream'.
Published batch 'batch_347' with 88 records to topic 'accident_stream'.
Published batch 'batch_348' with 77 records to topic 'accident_stream'.
Published batch 'batch_349' with 55 records to topic 'accident_stream'.
Published batch 'batch_350' with 98 records to topic 'accident_stream'.
Published batch 'batch_351' with 62 records to topic 'accident_stream'.
Published batch 'batch_352' with 93 records to topic 'accident_stream'.
Published batch 'batch_353' with 55 records to topic 'accident_stream'.
Published batch 'batch_354' with 56 records to topic 'accident_stream'.
Published batch 'batch_355' with 67 records to topic 'accident_stream'.
Published batch 'batch_356' with 84 records to topic 'accident_stream'.
Published batch 'batch_357' with 53 records to topic 'accident_